# overflow1: dense water flowing down a slope
pyOM2: for_config/overflow1.f90

Non-hydrostatic x-z slice (nh_realistic). Cold water left of x = L0/3 lies
next to warmer water. A Gaussian ridge at x = L0/3 reaches up to half the
depth. The cold water spreads over the ridge and flows down the slope on the
right as a gravity current, mixing with the ambient water.

In [ ]:
import sys; sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.nh_realistic.model import model

In [ ]:
fac        = 1.0
mix        = 5e-3
nx, ny, nz = int(128*fac), 1, int(40*fac)
dx = dz    = 0.5/fac
dt         = 0.25/fac
L0, H0     = nx*dx, nz*dz
nsteps     = int(8000*fac)            # 2000 s
plot_every = int(400*fac)

par = dict(nx=nx, ny=ny, nz=nz, dxt=dx, dyt=dx, dzt=dz, dt=dt,
           coord_degree=False, set_zero_at_surface=True,
           periodic_in_x=False, periodic_in_y=False,
           eq_of_state=1, implicit_free_surface=False, explicit_free_surface=False,
           epsilon=1e-12, max_iterations=5000, epsilon_nh=1e-6, superbee_advection=True,
           Ah=mix/fac**2, vmix_scheme='explicit', A_v=mix/fac**2)

In [ ]:
class overflow1(model):

    def set_topography(self):
        depth = H0 - H0/2*np.exp(-(self.xt - L0/3)**2/(L0/4)**2)
        kbot = np.ones((self.Ny, self.Nx))
        for k in range(nz):
            kbot[:, self.zt_gl[k] < -depth] = k + 1
        self.kbot = OM.np.asarray(kbot)

    def set_initial_conditions(self):
        T = 0.5 + np.zeros((nz, ny, nx))
        T[:, :, self.xt_gl <= L0/3] = 0.
        zeros = np.zeros((nz, ny, nx))
        super().set_initial_conditions(zeros, zeros, T, zeros, None)

In [ ]:
M = overflow1(par)
M.set_topography()
M.prepare_for_run()
M.set_initial_conditions()
h = M.halo
x, z = M.xt_gl, M.zt_gl

In [ ]:
def plot(t):
    clear_output(wait=True)
    mask = np.where(M.maskT[h:-h, h, h:-h] > 0, 1., np.nan)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
    for a, field, label in zip(ax, (M.temp, M.u, M.w),
                               ('temp [deg C]', 'u [m/s]', 'w [m/s]')):
        c = a.pcolormesh(x, z, field[h:-h, h, h:-h]*mask, cmap='RdBu_r')
        plt.colorbar(c, ax=a); a.set_title(label); a.set_xlabel('x [m]')
    ax[0].set_ylabel('z [m]')
    fig.suptitle('t = %.0f s' % t); plt.tight_layout(); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step_implicit_pressure(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
plot(nsteps*dt)